# PainNAS on Google Colab

This notebook runs the supervised early-fusion neural architecture search and, optionally, the full 87-fold BioVid LOSO evaluation. It mirrors the operational setup in `main.ipynb`: the repository is cloned into the Colab VM, BioVid is staged from Google Drive to the local SSD, the input contract is checked before training, and all durable experiment artifacts are written back to Drive.

Before running, select **Runtime → Change runtime type → GPU**. Place a BioVid archive such as `BioVid.tar.gz` in `/content/drive/MyDrive/PainData`, or provide an already extracted BioVid `PartA` directory in Drive. The Optuna database and completed LOSO folds are resumable after a disconnect.

## 1. Mount Drive and check out the repository

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/hhihn/FewShotPainAdaptation.git"
PROJECT_DIR = Path("/content/FewShotPainAdaptation")
BRANCH_NAME = "main"

if not PROJECT_DIR.exists():
    !git clone -b $BRANCH_NAME $REPO_URL $PROJECT_DIR
else:
    %cd $PROJECT_DIR
    !git pull --ff-only

%cd $PROJECT_DIR
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

assert (PROJECT_DIR / "painnas").is_dir(), PROJECT_DIR
print("Repository:", PROJECT_DIR)
print("Branch:", BRANCH_NAME)

## 2. Install the pinned experiment dependencies

In [ ]:
!pip -q install -U pip
!pip -q install -r $PROJECT_DIR/painnas/requirements-colab.txt
!pip -q install pandas matplotlib

## 3. Stage BioVid on the Colab SSD

Reading the full dataset repeatedly from mounted Drive is slow. The preferred path copies a tar archive to `/content` and extracts it under `/content/PainData`. The staging helper is idempotent within one runtime. If no supported archive exists, the cell falls back to an already extracted Drive directory.

In [ ]:
from pathlib import Path

from data_loaders.dataset_staging import stage_predefined_dataset_from_archive

DRIVE_DATA_DIR = Path("/content/drive/MyDrive/PainData")
LOCAL_DATA_DIR = Path("/content/PainData")
# Add a non-standard archive path here if necessary.
EXTRA_ARCHIVE_CANDIDATES = (
    DRIVE_DATA_DIR / "BioVid 2.tar.gz",
)
DRIVE_EXTRACTED_CANDIDATES = (
    DRIVE_DATA_DIR / "BioVid" / "PartA",
    DRIVE_DATA_DIR / "BioVid 2" / "PartA",
    DRIVE_DATA_DIR / "PartA",
)

try:
    BIOVID_ROOT = stage_predefined_dataset_from_archive(
        "biovid_part_a",
        drive_data_dir=DRIVE_DATA_DIR,
        local_data_dir=LOCAL_DATA_DIR,
        local_archive_dir=Path("/content"),
        extra_archive_candidates=EXTRA_ARCHIVE_CANDIDATES,
    )
    DATA_LOCATION = "local Colab SSD"
except FileNotFoundError as archive_error:
    BIOVID_ROOT = next(
        (
            candidate
            for candidate in DRIVE_EXTRACTED_CANDIDATES
            if (candidate / "Train").is_dir()
            and (candidate / "Test").is_dir()
        ),
        None,
    )
    if BIOVID_ROOT is None:
        raise archive_error
    DATA_LOCATION = "mounted Drive (slower fallback)"

DATA_DIR = BIOVID_ROOT
assert (DATA_DIR / "Train").is_dir(), DATA_DIR
assert (DATA_DIR / "Test").is_dir(), DATA_DIR
for modality in ("GSR", "ECG", "EMG"):
    assert (DATA_DIR / "Train" / modality).is_dir(), modality
    assert (DATA_DIR / "Test" / modality).is_dir(), modality

print("BioVid root:", DATA_DIR)
print("Data location:", DATA_LOCATION)

## 4. Verify the GPU and initialize reproducibility

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import platform
import random
import shutil
import numpy as np
import tensorflow as tf

GPUS = tf.config.list_physical_devices("GPU")
assert GPUS, "No TensorFlow GPU is visible. Select a Colab GPU runtime."
for gpu in GPUS:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

SEED = 42
tf.keras.utils.set_random_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
free_gib = shutil.disk_usage("/content").free / 1024**3

print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("Visible GPUs:", GPUS)
print(f"Free local disk: {free_gib:.1f} GiB")
print("Seed:", SEED)

## 5. Configure the NAS and LOSO run

The defaults run 50 Optuna trials for at most 50 epochs and then train the selected architecture from scratch in all 87 LOSO folds for at most 100 epochs. The exact Table 2 network is enqueued as trial 0. Change `RUN_NAME` when changing any configuration value; otherwise the manifest deliberately refuses an incompatible resume.

The architecture search is global rather than nested inside each LOSO fold. The resulting LOSO estimate is therefore exploratory, and every output manifest records this limitation.

In [ ]:
from datetime import datetime
import json

from painnas.config import PROTOCOL_WARNING, PainNASConfig
from painnas.io import atomic_write_json

RUN_NAME = "run_001"  # Keep stable to resume; change for a new configuration.
OUTPUT_ROOT = Path("/content/drive/MyDrive/PainNAS")
OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESUME = True
VERBOSE = 1
RUN_LOSO_AFTER_SEARCH = True
LOSO_START_INDEX = None  # One-based and inclusive.
LOSO_STOP_INDEX = None   # Use e.g. 1 and 30 to run a chunk.
MAX_FOLDS = None         # Debug only; keep None for the full experiment.

CONFIG = PainNASConfig(
    seed=SEED,
    batch_size=40,
    n_trials=50,
    search_max_epochs=50,
    loso_max_epochs=100,
    search_patience=8,
    loso_patience=15,
    search_validation_subjects=17,
    max_parameters=32_000_000,
    bootstrap_samples=10_000,
)

atomic_write_json(
    OUTPUT_DIR / "notebook_settings.json",
    {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "repository": str(PROJECT_DIR),
        "branch": BRANCH_NAME,
        "data_dir": str(DATA_DIR),
        "output_dir": str(OUTPUT_DIR),
        "config": CONFIG.to_dict(),
        "run_loso_after_search": RUN_LOSO_AFTER_SEARCH,
        "loso_start_index": LOSO_START_INDEX,
        "loso_stop_index": LOSO_STOP_INDEX,
        "max_folds": MAX_FOLDS,
        "protocol_warning": PROTOCOL_WARNING,
    },
)

print(json.dumps(CONFIG.to_dict(), indent=2))
print("Output directory:", OUTPUT_DIR)
print("Protocol warning:", PROTOCOL_WARNING)

## 6. Load and validate the real BioVid data

Only T0 and T4 are retained and remapped to binary labels. The expected input is 87 subjects, three modalities in GSR/ECG/EMG order, and 1152 time points. This cell also inspects the first LOSO fold before any training begins.

In [ ]:
import pandas as pd

from painnas.data import build_loso_fold_indices, load_biovid_binary

ARRAYS = load_biovid_binary(str(DATA_DIR), CONFIG)
first_subject = int(ARRAYS.unique_subjects[0])
first_fold = build_loso_fold_indices(ARRAYS, first_subject)

dataset_summary = pd.DataFrame(
    [
        {
            "samples": len(ARRAYS.y),
            "subjects": len(ARRAYS.unique_subjects),
            "sequence_length": ARRAYS.sequence_length,
            "modalities": ARRAYS.num_modalities,
            "t0_samples": int(np.sum(ARRAYS.y == 0)),
            "t4_samples": int(np.sum(ARRAYS.y == 1)),
            "train_samples": int(
                np.sum(ARRAYS.split_codes == ARRAYS.train_split_code)
            ),
            "test_samples": int(
                np.sum(ARRAYS.split_codes == ARRAYS.test_split_code)
            ),
        }
    ]
)
fold_summary = pd.DataFrame(
    [
        {
            "target_subject": ARRAYS.subject_keys[first_subject],
            "source_train_samples": len(first_fold.train),
            "source_validation_samples": len(first_fold.validation),
            "target_test_samples": len(first_fold.test),
        }
    ]
)
display(dataset_summary)
display(fold_summary)
print("Loaded tensor layout [samples, time, modalities]:", ARRAYS.X.shape)
print("CNN batches will be transposed to [batch, 3, 1152, 1].")

## 7. Inspect the exact Table 2 baseline

In [ ]:
from painnas.model import ArchitectureSpec, build_early_fusion_model

baseline_spec = ArchitectureSpec.baseline()
baseline_model = build_early_fusion_model(baseline_spec)
baseline_model.summary()
print(f"Baseline parameters: {baseline_model.count_params():,}")
del baseline_model
tf.keras.backend.clear_session()

## 8. Run or resume neural architecture search

This cell can be rerun after a disconnect. Trial state is stored in `search/study.sqlite3`, and completed/pruned/failed trials count toward the configured trial budget. A GPU out-of-memory candidate is recorded as failed without stopping the study.

In [ ]:
from painnas.search import run_search

SEARCH_RESULT = run_search(
    ARRAYS,
    CONFIG,
    OUTPUT_DIR / "search",
    resume=RESUME,
    verbose=VERBOSE,
)
display(pd.DataFrame([SEARCH_RESULT]))
print("Best architecture:", SEARCH_RESULT["best_architecture_path"])

## 9. Inspect NAS progress and the selected architecture

In [ ]:
import matplotlib.pyplot as plt

trials_path = OUTPUT_DIR / "search" / "trials.csv"
best_path = OUTPUT_DIR / "search" / "best_architecture.json"
trials = pd.read_csv(trials_path)
best_payload = json.loads(best_path.read_text(encoding="utf-8"))

display(trials.tail(20))
display(pd.DataFrame([best_payload["architecture"]]))
display(trials.groupby("state").size().rename("trials").to_frame())

completed = trials.loc[trials["state"] == "COMPLETE"].copy()
if not completed.empty:
    completed = completed.sort_values("trial_number")
    completed["best_so_far_macro_f1"] = completed["value"].cummax()
    ax = completed.plot(
        x="trial_number",
        y=["value", "best_so_far_macro_f1"],
        marker="o",
        figsize=(10, 4),
        title="PainNAS validation macro-F1",
    )
    ax.set_xlabel("trial")
    ax.set_ylabel("macro-F1")
    ax.set_ylim(0, 1)
    plt.show()

print(json.dumps(best_payload, indent=2))

## 10. Optionally run or resume LOSO

Each fold clears Keras state and creates a new network and optimizer. NAS weights are never reused. Set `RUN_LOSO_AFTER_SEARCH=False` in the configuration cell when this notebook should stop after architecture search. Fold ranges can be used to distribute the 87 folds across Colab sessions while keeping the same output directory.

In [ ]:
from painnas.loso import run_loso
from painnas.search import load_architecture

LOSO_RESULT = None
if RUN_LOSO_AFTER_SEARCH:
    selected_spec = load_architecture(
        OUTPUT_DIR / "search" / "best_architecture.json"
    )
    LOSO_RESULT = run_loso(
        ARRAYS,
        selected_spec,
        CONFIG,
        OUTPUT_DIR / "loso",
        resume=RESUME,
        start_index=LOSO_START_INDEX,
        stop_index=LOSO_STOP_INDEX,
        max_folds=MAX_FOLDS,
        verbose=VERBOSE,
    )
    print(json.dumps(LOSO_RESULT, indent=2))
else:
    print("LOSO skipped. The NAS outputs remain in", OUTPUT_DIR / "search")

## 11. Summarize accumulated LOSO results

In [ ]:
fold_metrics_path = OUTPUT_DIR / "loso" / "fold_metrics.csv"
summary_path = OUTPUT_DIR / "loso" / "summary.json"

if fold_metrics_path.exists():
    fold_metrics = pd.read_csv(fold_metrics_path)
    display(fold_metrics)
    ax = fold_metrics.plot(
        x="fold_index",
        y=["accuracy", "macro_f1"],
        marker="o",
        figsize=(12, 4),
        title="Accumulated BioVid LOSO metrics",
    )
    ax.set_xlabel("LOSO fold")
    ax.set_ylabel("score")
    ax.set_ylim(0, 1)
    plt.show()
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    display(pd.DataFrame(summary["metrics"]).T)
    print("Completed folds:", summary["completed_folds"])

print("Durable artifacts:", OUTPUT_DIR)

## 12. Optional runtime cleanup

Run this only after the desired NAS/LOSO work has finished. All durable artifacts already reside in Drive.

In [ ]:
import gc
import logging

try:
    del ARRAYS
except NameError:
    pass
tf.keras.backend.clear_session()
gc.collect()
logging.shutdown()

try:
    from google.colab import runtime
except ImportError:
    print("Cleanup complete; no hosted Colab runtime was detected.")
else:
    print("Cleanup complete; disconnecting and deleting the Colab runtime.")
    runtime.unassign()